<i>Copyright (c) Recommenders contributors.</i>

<i>Licensed under the MIT License.</i>

# DKN : Deep Knowledge-Aware Network for News Recommendation

DKN \[1\] is a deep learning model which incorporates information from knowledge graph for better news recommendation. Specifically, DKN uses TransX \[2\] method for knowledge graph representation learning, then applies a CNN framework, named KCNN, to combine entity embedding with word embedding and generate a final embedding vector for a news article. CTR prediction is made via an attention-based neural scorer. 

## Properties of DKN:

- DKN is a content-based deep model for CTR prediction rather than traditional ID-based collaborative filtering. 
- It makes use of knowledge entities and common sense in news content via joint learning from semantic-level and knowledge-level representations of news articles.
- DKN uses an attention module to dynamically calculate a user's aggregated historical representaition.


## CPU first run and input format

The default uses invented articles, users and impressions with random 16-dimensional embeddings. It checks model execution with reduced dimensions; real MIND performance is unmeasured. [Installation, trusted prepared inputs and results](NEWS_CPU_VALIDATION.md).

DKN reads indexed `label userid candidate%impression` records, a per-user history file and aligned article word/entity index arrays. Three NumPy matrices supply word, entity and context embeddings. This fixture uses random matrices, including the entity/context channels. Set `DKN_DATA_DIR` to a trusted compatible prepared folder to supply real inputs.


## Global settings and imports

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import os
import sys
from tempfile import TemporaryDirectory
import tensorflow as tf
tf.get_logger().setLevel("ERROR") # only show error messages
tf.compat.v1.logging.set_verbosity(tf.compat.v1.logging.ERROR)

from recommenders.models.deeprec.deeprec_utils import download_deeprec_resources, prepare_hparams
from recommenders.models.deeprec.models.dkn import DKN
from recommenders.models.deeprec.io.dkn_iterator import DKNTextIterator
from recommenders.utils.notebook_utils import store_metadata

print(f"System version: {sys.version}")
print(f"Tensorflow version: {tf.__version__}")
from recommenders.models.newsrec.quickstart import make_synthetic_news, require_news_files

import random
random.seed(42)


## Input selection
The default creates synthetic indexed articles, random word/entity/context embeddings and invented interactions in a temporary directory. Set `DKN_DATA_DIR` to a trusted prepared DKN folder to use its original input files and YAML. No real MIND DKN run is claimed here.


In [ ]:
tmpdir = TemporaryDirectory()
prepared = os.environ.get("DKN_DATA_DIR")
if prepared:
    data_path = require_news_files(prepared, ["dkn.yaml", "train_mind_demo.txt", "valid_mind_demo.txt", "test_mind_demo.txt", "doc_feature.txt", "user_history.txt", "word_embeddings_100.npy", "TransE_entity2vec_100.npy", "TransE_context2vec_100.npy"])
    data_kind = "local_prepared_unverified_provenance"
else:
    data_path = os.path.join(str(make_synthetic_news(tmpdir.name)), "mind-demo-dkn")
    data_kind = "synthetic"
print("Input kind:", data_kind, "| GPU devices:", tf.config.list_physical_devices("GPU"))
store_metadata("data_kind", data_kind)
yaml_file = os.path.join(data_path, "dkn.yaml")
train_file = os.path.join(data_path, "train_mind_demo.txt")
valid_file = os.path.join(data_path, "valid_mind_demo.txt")
test_file = os.path.join(data_path, "test_mind_demo.txt")
news_feature_file = os.path.join(data_path, "doc_feature.txt")
user_history_file = os.path.join(data_path, "user_history.txt")
wordEmb_file = os.path.join(data_path, "word_embeddings_100.npy")
entityEmb_file = os.path.join(data_path, "TransE_entity2vec_100.npy")
contextEmb_file = os.path.join(data_path, "TransE_context2vec_100.npy")

store_metadata("gpu_count", len(tf.config.list_physical_devices("GPU")))


## Create hyper-parameters

In [ ]:
EPOCHS = int(os.environ.get("NEWS_EPOCHS", "1"))
BATCH_SIZE = 8
assert EPOCHS > 0


In [ ]:
hparams = prepare_hparams(yaml_file,
                          news_feature_file = news_feature_file,
                          user_history_file = user_history_file,
                          wordEmb_file=wordEmb_file,
                          entityEmb_file=entityEmb_file,
                          contextEmb_file=contextEmb_file,
                          epochs=EPOCHS,
                          save_model=False,
                          batch_size=BATCH_SIZE)
print(hparams)

## Train the DKN model

In [ ]:
model = DKN(hparams, DKNTextIterator)

In [ ]:
print(model.run_eval(valid_file))

In [ ]:
model.fit(train_file, valid_file)

## Evaluate the DKN model

Now we can check the performance on the test set:

In [ ]:
res = model.run_eval(test_file)
print(res)

In [ ]:
# Record results for tests - ignore this cell
store_metadata("auc", res["auc"])
store_metadata("group_auc", res["group_auc"])
store_metadata("ndcg@5", res["ndcg@5"])
store_metadata("ndcg@10", res["ndcg@10"])
store_metadata("mean_mrr", res["mean_mrr"])

checkpoint = os.path.join(tmpdir.name, "dkn-checkpoint")
with model.graph.as_default():
    saver = tf.compat.v1.train.Saver()
    saver.save(model.sess, checkpoint)
    saver.restore(model.sess, checkpoint)
restored = model.run_eval(test_file)
assert all(abs(res[k] - restored[k]) < 1e-6 for k in res)
store_metadata("reload_metrics_equal", True)
model.sess.close()


## References

\[1\] Wang, Hongwei, et al. "DKN: Deep Knowledge-Aware Network for News Recommendation." Proceedings of the 2018 World Wide Web Conference on World Wide Web. International World Wide Web Conferences Steering Committee, 2018.<br>
\[2\] Knowledge Graph Embeddings including TransE, TransH, TransR and PTransE. https://github.com/thunlp/KB2E <br>
\[3\] Wu, Fangzhao, et al. "MIND: A Large-scale Dataset for News Recommendation" Proceedings of the 58th Annual Meeting of the Association for Computational Linguistics. https://msnews.github.io/competition.html <br>
\[4\] GloVe: Global Vectors for Word Representation. https://nlp.stanford.edu/projects/glove/

In [ ]:
tmpdir.cleanup()
